# Studying the effect of pooling - Design A (scaled-down pool)

This is pooled data but reduced to the size of one observer's data (about 1800 trials).

Pooling the data of the five observers O1 - O5 (pooled-5x, built in `finding_dimensionality_aggregated_per_observer.ipynb`) gives a higher dimensionality than the mean of the individual observers in four of the six conditions (task x image set), and never a lower one. Pooled-5x, however, holds five times the data of one observer. This notebook asks whether the higher dimensionality is explained by the amount of data alone.

To answer it, we build pooled data that differ from pooled-5x in size only. We keep a random fifth of the trials of every session of pooled-5x  (design A, scaled-down pool; label `design-A`). The notebook runs all three image sets.

In [1]:
import ast
import os

import numpy as np
import pandas as pd
import sklearn

import cblearn

from load_data import load_all
from dimensionality_estimation import estimate_dimensionality
from io_json import read_json_dataclass, save_json_dataclass
from utils import read_dimensionality
from size_controlled_pooling import subsample_sessions, to_triplets

# seeds the cross-validation splits and the SOE initialisations of every fit;
# draw k of the subsample is seeded with SEED + k
SEED = 0
# number of independent subsamples
N_DRAWS = 10
# share of the trials of every session that the subsample keeps
FRACTION = 1 / 5
LABEL = 'design-A'
# all fitted runs; read back instead of refitting when it and the jsons exist
RUNS_CSV = '../data/found-dimensionality-embeddings-design-A-draws.csv'

OBSERVERS = ['O1', 'O2', 'O3', 'O4', 'O5']
TASKS = ['classic', 'odd one out']
IMAGE_SETS = ['action', 'greebles', 'taxonomic']

print('cblearn:', cblearn.__version__)
print('sklearn:', sklearn.__version__)

cblearn: 0.3.0
sklearn: 1.5.2


# Reading data

`load_all(fill_missing_O_by_P=True)` relabels the sessions that O5 did under the pseudonym P1 (1a, 1b, 2a, 2b) as O5. Keeping O1 - O5 gives, for every task and image set, the 30 sessions of pooled-5x (5 observers x 2 trial sets x 3 repeats). The cell checks that these sessions hold the same number of trials as the original pooled data.

In [2]:
triplet_df = load_all(fill_missing_O_by_P=True)
triplet_df = triplet_df[triplet_df['observer'].isin(OBSERVERS)]

pooled_5x = pd.read_csv('../data/found-dimensionality-embeddings-pooled-observers-nsplit-5-nrepeat-30.csv')
n_trials_5x = {}
for task in TASKS:
    for image_set in IMAGE_SETS:
        condition = triplet_df[(triplet_df['task'] == task) & (triplet_df['image_set'] == image_set)]
        assert len(condition) == 30
        n_trials = condition['responses'].apply(len).sum()
        responses_5x = ast.literal_eval(pooled_5x.query('task == @task and image_set == @image_set')['responses'].item())
        assert n_trials == len(responses_5x)
        n_trials_5x[task, image_set] = n_trials

pd.Series(n_trials_5x, name='trials in pooled-5x')

classic      action       8846
             greebles     8839
             taxonomic    8842
odd one out  action       8997
             greebles     8998
             taxonomic    8997
Name: trials in pooled-5x, dtype: int64

# Design

For every task and image set, pooled-5x stacks all trials of 30 sessions: 5 observers x 2 trial sets x 3 repeats, about 9000 trials. Every triplet of a trial list is answered 15 times, three times by each observer. A triplet here is one question in one screen layout (`triplet_index`).

For every task and image set, we have 30 sessions (5 observers * 3 repeats * 2 trial sets), each consisting of 300 trials.
We draw a random fifth of the trials (without replacement) of each of those sessions, these are 60 of 300 trials.
Sessions shortened by timeouts keep a fifth of the trials they have, rounded to the nearest integer (30 of the one classic session with 150 trials). 
After drawing from all sessions, trials were pooled together in a subsample. 
The subsample then holds 60 trials * 5 observers * 2 trial sets * 3 repeats = 1800 trials,
the size of one observer's data, with about 60 * 2 * 3 = 360 trials from each observer.

Drawing the same fraction from every session keeps the share of every observer, trial set and repeat as in pooled-5x, up to rounding. Within a session, the kept trials are a simple random sample.
The subsample keeps the repeats of each trial on purpose, because pooled-5x contains them.


We draw the subsample `N_DRAWS = 10` times, with seed `SEED + k`, where $k$ is the draw number. Each draw, task and image set has its own random generator, seeded with `SEED + k` together with the numbers of the task and the image set, so that the subsample of a condition does not depend on which other conditions are run. We fit every draw and report the mean dimensionality over the 10 draws. All fits use the same seeds for the cross-validation splits and the SOE initialisations, so the draws differ only in the subsample.

In [3]:
runs_records = []  # one row per draw, task and image set: the data that is fitted

for draw in range(N_DRAWS):
    for task in TASKS:
        for image_set in IMAGE_SETS:
            # one generator per draw and condition, so that a condition's subsample does
            # not depend on which other conditions are run
            rng = np.random.default_rng([SEED + draw, TASKS.index(task), IMAGE_SETS.index(image_set)])
            condition = triplet_df[(triplet_df['task'] == task) & (triplet_df['image_set'] == image_set)]
            record = subsample_sessions(condition, FRACTION, rng)
            runs_records.append({'observer': LABEL, 'draw': draw, 'seed': SEED + draw,
                                 'task': task, 'image_set': image_set,
                                 'n_trials': len(record['responses']), **record,
                                 'embedding_ready_triplets': to_triplets(task, record['raw_triplets'],
                                                                         record['responses'])})

runs_df = pd.DataFrame(runs_records)
runs_df['n_triplets'] = runs_df['embedding_ready_triplets'].apply(len)
runs_df[['draw', 'task', 'image_set', 'n_trials', 'n_triplets']]

,draw,task,image_set,n_trials,n_triplets
0,0,classic,action,1770,1770
1,0,classic,greebles,1769,1769
2,0,classic,taxonomic,1769,1769
3,0,odd one out,action,1800,3600
4,0,odd one out,greebles,1800,3600
5,0,odd one out,taxonomic,1800,3600
6,1,classic,action,1770,1770
7,1,classic,greebles,1769,1769
8,1,classic,taxonomic,1769,1769
9,1,odd one out,action,1800,3600


In [4]:
runs_df

,observer,draw,seed,task,image_set,n_trials,raw_triplets,responses,triplet_index,source_observer,source_session,embedding_ready_triplets,n_triplets
0,design-A,0,0,classic,action,1770,"[[11, 8, 17], [0, 5, 9], [9, 6, 16], [5, 15, 9...","[1, 1, 1, -1, -1, -1, 1, -1, -1, -1, -1, 1, 1,...","[14, 19, 18, 46, 36, 49, 40, 42, 10, 4, 2, 16,...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[11, 8, 17], [0, 5, 9], [9, 6, 16], [5, 9, 15...",1770
1,design-A,0,0,classic,greebles,1769,"[[6, 11, 12], [5, 3, 15], [0, 3, 1], [1, 0, 17...","[1, 1, -1, 1, -1, -1, -1, 1, 1, 1, 1, -1, -1, ...","[29, 24, 1, 42, 49, 32, 22, 18, 40, 17, 35, 89...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[6, 11, 12], [5, 3, 15], [0, 1, 3], [1, 0, 17...",1769
2,design-A,0,0,classic,taxonomic,1769,"[[4, 7, 14], [1, 14, 11], [3, 0, 10], [2, 13, ...","[1, -1, -1, 1, 1, -1, 1, 1, 1, 1, -1, 1, 1, -1...","[22, 49, 39, 20, 74, 71, 94, 99, 61, 54, 69, 5...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[4, 7, 14], [1, 11, 14], [3, 10, 0], [2, 13, ...",1769
3,design-A,0,0,odd one out,action,1800,"[[15, 3, 7], [14, 11, 13], [7, 9, 13], [6, 9, ...","[1, 1, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, ...","[4, 33, 16, 18, 36, 32, 40, 23, 41, 38, 5, 64,...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1...","[[15, 7, 3], [14, 13, 11], [7, 9, 13], [9, 16,...",3600
4,design-A,0,0,odd one out,greebles,1800,"[[4, 13, 15], [5, 8, 9], [10, 5, 7], [8, 0, 2]...","[0, 0, 1, 0, 0, 1, 2, 1, 0, 2, 2, 1, 2, 1, 2, ...","[15, 10, 18, 6, 45, 44, 17, 11, 1, 30, 46, 67,...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1...","[[13, 15, 4], [8, 9, 5], [10, 7, 5], [0, 2, 8]...",3600
5,design-A,0,0,odd one out,taxonomic,1800,"[[6, 3, 10], [12, 1, 9], [16, 5, 14], [12, 10,...","[0, 1, 2, 1, 1, 1, 0, 1, 2, 0, 1, 2, 0, 2, 2, ...","[9, 43, 6, 21, 8, 34, 13, 27, 32, 3, 24, 17, 2...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1...","[[3, 10, 6], [12, 9, 1], [16, 5, 14], [12, 13,...",3600
6,design-A,1,1,classic,action,1770,"[[0, 13, 3], [1, 10, 13], [8, 16, 10], [3, 11,...","[-1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1...","[31, 49, 22, 13, 4, 7, 29, 16, 9, 1, 96, 93, 7...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[0, 3, 13], [1, 13, 10], [8, 16, 10], [3, 17,...",1770
7,design-A,1,1,classic,greebles,1769,"[[1, 15, 5], [5, 3, 15], [9, 16, 15], [10, 16,...","[-1, 1, -1, -1, -1, 1, -1, -1, 1, -1, 1, -1, 1...","[45, 24, 48, 5, 13, 28, 85, 69, 56, 93, 86, 95...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[1, 5, 15], [5, 3, 15], [9, 15, 16], [10, 15,...",1769
8,design-A,1,1,classic,taxonomic,1769,"[[3, 8, 17], [2, 4, 12], [3, 2, 10], [7, 13, 1...","[1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, -1, ...","[16, 40, 18, 15, 24, 3, 6, 21, 33, 19, 48, 0, ...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1a, 1...","[[3, 8, 17], [2, 12, 4], [3, 10, 2], [7, 14, 1...",1769
9,design-A,1,1,odd one out,action,1800,"[[14, 11, 13], [17, 3, 5], [9, 10, 17], [5, 4,...","[1, 0, 1, 2, 1, 1, 0, 0, 1, 1, 2, 2, 1, 2, 1, ...","[33, 21, 25, 48, 46, 35, 8, 10, 14, 26, 64, 71...","[O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O1, O...","[1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1b, 1...","[[14, 13, 11], [3, 5, 17], [9, 17, 10], [5, 4,...",3600


Number of trials that every observer contributes to the subsample of draw 0, next to one fifth of pooled-5x. The design gives about 360 per observer; fewer means that the observer's sessions were shortened by timeouts.

In [5]:
draw0 = runs_df[runs_df['draw'] == 0]
pd.DataFrame([{'task': row.task, 'image_set': row.image_set,
               **pd.Series(row.source_observer).value_counts().reindex(OBSERVERS).to_dict(),
               'total': row.n_trials, 'pooled-5x / 5': n_trials_5x[row.task, row.image_set] / 5}
              for row in draw0.itertuples()])

,task,image_set,O1,O2,O3,O4,O5,total,pooled-5x / 5
0,classic,action,360,360,360,330,360,1770,1769.2
1,classic,greebles,360,360,360,330,359,1769,1767.8
2,classic,taxonomic,360,360,360,330,359,1769,1768.4
3,odd one out,action,360,360,360,360,360,1800,1799.4
4,odd one out,greebles,360,360,360,360,360,1800,1799.6
5,odd one out,taxonomic,360,360,360,360,360,1800,1799.4


# Estimating dimensionality of every draw

Each draw is fitted with the significance criterion of `cblearn`. The cross-validation results of every fit are saved in `../data/triplet_accuracy_vs_dimensionality-design-A-draw{k}-{task}-{image_set}.json`. The embedding is fitted at the dimensionality of the significance criterion.

The fits run only when the runs table (`RUNS_CSV`) or the json of a fit is missing; the table is then saved. Otherwise the cell reads them back, after checking that the rebuilt pooling equals the saved one and that the table agrees with the jsons. Delete the table to refit.

In [6]:
# the fits run when the runs table or the json of any fit is missing; 
# otherwise they are read back, the dimensionalities from the jsons, the embeddings from the table
json_names = [f"../data/triplet_accuracy_vs_dimensionality-{d['observer']}-draw{d['draw']}-{d['task']}-{d['image_set']}.json"
              for _, d in runs_df.iterrows()]
loaded = os.path.exists(RUNS_CSV) and all(os.path.exists(name) for name in json_names)

if loaded:
    saved = pd.read_csv(RUNS_CSV)
    assert list(saved[['draw', 'task', 'image_set']].itertuples(index=False)) == \
        list(runs_df[['draw', 'task', 'image_set']].itertuples(index=False)), 'the saved runs are in another order'
    # the pooling rebuilt above must be the one that was fitted
    assert all(np.array_equal(np.array(ast.literal_eval(s)), t)
               for s, t in zip(saved['embedding_ready_triplets'], runs_df['embedding_ready_triplets'])), \
        'the rebuilt pooling differs from the saved one'
    dimensionality = [read_json_dataclass(name).estimated_dimension for name in json_names]
    assert dimensionality == list(saved['dimensionality'])
    embedding = [ast.literal_eval(s) for s in saved['embedding']]
    print(f'read {len(json_names)} saved fits and {RUNS_CSV}')
else:
    dimensionality = []
    embedding = []

    for ix, d in runs_df.iterrows():
        obs, draw, task, image_set = d['observer'], d['draw'], d['task'], d['image_set']
        triplets = d['embedding_ready_triplets']

        dim_result, fitted_embedding = estimate_dimensionality(triplets, random_state=SEED,
                                                               soe_random_state=SEED)
        assert fitted_embedding.shape[1] == dim_result.estimated_dimension

        # saving triplet accuracy and statistics
        save_json_dataclass(dim_result, f"../data/triplet_accuracy_vs_dimensionality-{obs}-draw{draw}-{task}-{image_set}.json")

        dimensionality.append(dim_result.estimated_dimension)
        embedding.append(fitted_embedding.tolist())

        print(f"{obs} - draw {draw} - {task} - {image_set} - {len(triplets)} triplets: "
              f"{dim_result.estimated_dimension} dimension(s)")

runs_df['dimensionality'] = dimensionality
runs_df['embedding'] = embedding

read 60 saved fits and ../data/found-dimensionality-embeddings-design-A-draws.csv


In [7]:
if not loaded:
    # all draws; list-valued columns are saved as their repr, read back with ast.literal_eval
    to_save = runs_df.copy()
    for column in ['raw_triplets', 'responses', 'triplet_index', 'source_observer', 'source_session',
                   'embedding_ready_triplets', 'embedding']:
        to_save[column] = to_save[column].apply(lambda x: np.asarray(x).tolist())
    to_save.to_csv(RUNS_CSV, index=False)

# Summary

For each task and image set: the dimensionality of every observer and their mean, of pooled-5x, and the mean dimensionality over the 10 draws of design A.

In [8]:
# the individual observers and pooled-5x are read from their jsons (read_dimensionality)
mean_a = runs_df.groupby(['task', 'image_set'])['dimensionality'].mean()

rows = []
for task in TASKS:
    for image_set in IMAGE_SETS:
        row = {'task': task, 'image_set': image_set}
        for obs in OBSERVERS:
            row[obs] = read_dimensionality(obs, task, image_set)
        row['mean O1 - O5'] = np.mean([row[obs] for obs in OBSERVERS])
        row['pooled-5x'] = read_dimensionality('pooled O1 - O5', task, image_set)
        row['design A, mean of 10 draws'] = mean_a[task, image_set]
        rows.append(row)

summary = pd.DataFrame(rows)
with pd.option_context('display.precision', 1):
    display(summary)

,task,image_set,O1,O2,O3,O4,O5,mean O1 - O5,pooled-5x,"design A, mean of 10 draws"
0,classic,action,2,1,2,2,1,1.6,4,1.0
1,classic,greebles,2,2,1,2,3,2.0,2,2.0
2,classic,taxonomic,3,2,2,1,2,2.0,2,1.3
3,odd one out,action,2,1,3,2,1,1.8,3,1.0
4,odd one out,greebles,1,2,1,2,2,1.6,2,2.1
5,odd one out,taxonomic,2,2,1,1,1,1.4,2,1.4
